In [ ]:
# --- Vishalam setup: find the code dataset among the inputs, unpack, install (no secrets) ---
import glob, os, shutil, subprocess, sys, tarfile, time
T0 = time.time()
def hours_left(limit=11.2):  # Kaggle stops sessions at 12 h; keep a margin for saving outputs
    return max(0.25, limit - (time.time() - T0) / 3600)
WORK, INPUT, CODE = "/kaggle/working", "/kaggle/input", "/tmp/vishalam"

def inputs_with(marker, hint=""):
    """Input directories that contain `marker` (a relative path) and whose path contains `hint`."""
    hits = glob.glob(f"{INPUT}/**/{marker}", recursive=True)
    dirs = [h[: -len(marker)].rstrip("/") for h in hits]
    return sorted(d for d in set(dirs) if hint in d)

if not os.path.isdir(CODE):
    tars = glob.glob(f"{INPUT}/**/vishalam_code.tar.gz", recursive=True)
    if tars:
        os.makedirs(CODE)
        with tarfile.open(tars[0]) as t:
            t.extractall(CODE, filter="data")
    else:  # Kaggle sometimes unpacks archives itself
        roots = [d for d in inputs_with("vishalam/tokenizer/bpe.py") if os.path.exists(f"{d}/requirements.txt")]
        if not roots:
            raise SystemExit("code dataset not attached - run tools/kaggle_driver.py push-code and re-push this kernel")
        shutil.copytree(roots[0], CODE)
os.chdir(CODE)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("ready:", open("VERSION").read().strip() if os.path.exists("VERSION") else "code unpacked")


In [ ]:
# --- Test the TPU (PyTorch/XLA) training path on XLA's CPU backend: no TPU queue, no quota ---
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.8.0", "--index-url",
                "https://download.pytorch.org/whl/cpu"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch_xla==2.8.0"], check=True)
for n in ("1", "2"):  # one simulated chip first, then two (collectives)
    env = {**os.environ, "PJRT_DEVICE": "CPU", "CPU_NUM_DEVICES": n}
    r = subprocess.run([sys.executable, "-m", "pytest", "-x", "-v", "-p", "no:cacheprovider",
                        "--basetemp", f"/tmp/xla{n}", "tests/test_pretrain_xla.py"],
                       capture_output=True, text=True, env=env)
    print(f"==== CPU_NUM_DEVICES={n}", r.stdout[-4000:], sep="\n")
    for f in glob.glob(f"/tmp/xla{n}/**/last_failure.txt", recursive=True):
        text = open(f).read()
        print(f"---- {f}", text[:3000], "[...]", text[-6000:], sep="\n")
    print(f"XLA TESTS ({n} device) " + ("PASSED" if r.returncode == 0 else "FAILED"))
